# Atelier Python Session 4 : démo de l'API d'Arvest

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stage-to-data/hackspace-shs/blob/main/cours-python-resources/session-4/Atelier%20Python%20Session%204%20-%20D%C3%A9mo%20API%20Arvest.ipynb)

[Arvest](https://arvest.app) est une plateforme pour rassembler, annoter et exposer des images et des documents
patrimoniaux, notamment grâce au standard [IIIF](https://iiif.io/) (*International Image Interoperability Framework*).
Le package Python [`arvestapi`](https://github.com/arvest-data-in-context/arvest-api) permet d'interagir avec son API :
lister ses contenus, mais aussi en **ajouter** (médias, manifestes IIIF, projets), ce que nous n'avons pas fait avec
Nakala.

> ⚠ Pour suivre cette démo, il faut un compte Arvest. Les cellules des parties 3 à 6 **modifient votre compte** (elles
> ajoutent des médias, des manifestes et un projet) : vous pourrez les supprimer ensuite depuis l'interface d'Arvest.

Commençons par installer le package, directement depuis GitHub (`-q` rend l'installation silencieuse), et par
l'importer :

In [ ]:
!pip install -q git+https://github.com/arvest-data-in-context/arvest-api.git
import arvestapi

## 1. Se connecter à Arvest

Pour se connecter, il faut donner son email et son mot de passe.

> 🔒 **N'écrivez jamais un mot de passe directement dans un notebook** ! Il resterait visible par toutes les personnes
> avec qui vous partagez le notebook (ou qui le voient sur GitHub). Ici, on utilise `input()` pour l'email, et la
> fonction `getpass()` pour le mot de passe : elle fonctionne comme `input()`, mais masque ce que l'on tape.

In [ ]:
from getpass import getpass

email = input("Email Arvest : ")
mot_de_passe = getpass("Mot de passe Arvest : ")

ar = arvestapi.Arvest(email, mot_de_passe)

print("Connecté·e en tant que", ar.profile.name)

`ar` est un objet de la classe `Arvest`, qui représente notre connexion. Toutes les opérations passent par ses méthodes.

## 2. Lister ses contenus

Les méthodes `get_medias()`, `get_manifests()` et `get_projects()` renvoient des listes d'objets, qui ont chacun des
attributs comme `title` ou `description` :

In [ ]:
medias = ar.get_medias()
print(len(medias), "médias :")
for media in medias:
  print("-", media.title, ":", media.url)

manifestes = ar.get_manifests()
print(len(manifestes), "manifestes :")
for manifeste in manifestes:
  print("-", manifeste.title)

projets = ar.get_projects()
print(len(projets), "projets :")
for projet in projets:
  print("-", projet.title)

## 3. Ajouter des médias

`add_media()` ajoute un média, à partir de l'argument `path`, qui peut être :
- l'**URL** d'une image en ligne ;
- le **chemin** d'un fichier sur votre ordinateur (dans Colab : un fichier que vous avez déposé dans le panneau
  « Fichiers » à gauche).

Elle renvoie un objet `Media`, dont on peut ensuite modifier le titre et la description.

In [ ]:
import os

# Un média en ligne :
url_image = "https://upload.wikimedia.org/wikipedia/commons/a/af/Notre-Dame_de_Paris_2013-07-24.jpg"
nouveau_media = ar.add_media(path=url_image)
nouveau_media.update_title("Notre-Dame de Paris")
nouveau_media.update_description("Ajouté depuis Python !")

# Un fichier local : remplacez "mon_image.jpg" par le nom d'une image que vous avez déposée dans Colab.
chemin_local = os.path.join(os.getcwd(), "mon_image.jpg")
if os.path.isfile(chemin_local):
  media_local = ar.add_media(path=chemin_local)
  media_local.update_title("Mon image")
else:
  print("Pas de fichier", chemin_local, ": déposez d'abord une image dans le panneau Fichiers.")

## 4. Créer un projet

In [ ]:
nouveau_projet = ar.create_project(title="Mon projet créé avec Python")
print(nouveau_projet.title)

## 5. Les métadonnées

Chaque média peut avoir des métadonnées, par exemple au format Dublin Core. `get_metadata()` renvoie une liste avec un
élément par format de métadonnées :

In [ ]:
for media in ar.get_medias():
  if media.title == "Notre-Dame de Paris":
    for format_meta in media.get_metadata():
      print(format_meta["title"])
      print(format_meta["metadata"])

## 6. Les manifestes IIIF

Un **manifeste IIIF** est un fichier JSON qui décrit un objet patrimonial (un livre, une carte, une série de
photographies...) : ses métadonnées, et la liste de ses images, servies par un serveur d'images IIIF. De nombreuses
institutions publient leurs collections avec des manifestes IIIF, comme Gallica (BnF) : chaque document numérisé de
Gallica a un manifeste.

Comme `add_media()`, `add_manifest()` prend un argument `path`, qui peut être l'URL d'un manifeste en ligne, ou le chemin
d'un fichier JSON sur votre ordinateur.

In [ ]:
url_manifeste = "https://gallica.bnf.fr/iiif/ark:/12148/btv1b55014276q/manifest.json"

manifeste_lie = ar.add_manifest(path=url_manifeste)
manifeste_lie.update_title("Manifeste Gallica")

### Créer son propre manifeste avec `iiif-prezi3`

On peut aussi fabriquer un manifeste de toutes pièces. Le package
[`iiif-prezi3`](https://github.com/iiif-prezi/iiif-prezi3) permet de le faire en Python, sans écrire le JSON à la main :
on crée un objet `Manifest`, puis on lui ajoute des **canvas** (des « pages », chacune avec une image).

Ici, on crée un manifeste d'une page, à partir d'une image d'exemple hébergée sur un serveur IIIF, on l'enregistre dans
un fichier JSON, puis on l'envoie sur Arvest.

In [ ]:
!pip install -q iiif-prezi3

In [ ]:
import iiif_prezi3

manifeste = iiif_prezi3.Manifest(
  id="https://example.org/iiif/mon-manifeste.json",
  label={"fr": ["Mon premier manifeste"]}
)

# make_canvas_from_iiif() va chercher les dimensions de l'image sur le serveur IIIF, et crée la page :
manifeste.make_canvas_from_iiif(
  url="https://iiif.io/api/image/3.0/example/reference/918ecd18c2592080851777620de9bcb5-gottingen",
  id="https://example.org/iiif/canvas/1",
  label="Page 1"
)

# .json() renvoie le manifeste sous forme de string JSON :
print(manifeste.json(indent=2))

chemin_manifeste = os.path.join(os.getcwd(), "mon_manifeste.json")
with open(chemin_manifeste, "w", encoding="utf-8") as f:
  f.write(manifeste.json(indent=2))

In [ ]:
nouveau_manifeste = ar.add_manifest(path=chemin_manifeste)
nouveau_manifeste.update_title("Mon premier manifeste")